## ML models for Raman Spectroscopy

In [ ]:
#imports

In [ ]:
import pickle
import pandas as pd
import numpy as np
import yaml
import os
import math 
from glob import glob
import sys
 
import time
from rich.progress import track
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score 
from scipy.stats import norm


from sklearn.metrics import confusion_matrix

import xgboost as xgb
from rich.pretty import pprint

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [ ]:
path_to_data = './Raman_ML/data/18062026/'

In [ ]:
gt_path = './Raman_ML/data/ground_truth.csv'

#### Prepare Data

In [78]:
import os
import glob
import re
import pandas as pd
import numpy as np


def extract_id(filename):
    match = re.search(r'Scan\s+(\d+)', filename)
    return str(int(match.group(1))) if match else None


def read_raman_csv(filepath):
    try:
        df = pd.read_csv(filepath, header=None)
        if df.shape[1] < 2:
            raise ValueError("Invalid format")
        return df.iloc[:, 1].values  # only intensity
    except Exception as e:
        print(f"[ERROR] {filepath}: {e}")
        return None


def process_folder(folder_path):
    
    files = glob.glob(os.path.join(folder_path, "*.csv"))
     
    all_data = []
    max_len = 0

    # First pass: read + find max length
    temp = []
    for f in files:
        fname = os.path.basename(f)
        fid = extract_id(fname)

        if fid is None:
            continue

        intensity = read_raman_csv(f)
        if intensity is None:
            continue

        max_len = max(max_len, len(intensity))

        temp.append((fid, fname, intensity))

    print(f"[INFO] max spectrum length = {max_len}")

    # Second pass: pad + build rows
    for fid, fname, intensity in temp:
        padded = np.full(max_len, np.nan)
        padded[:len(intensity)] = intensity

        row = [fid, fname] + padded.tolist()
        all_data.append(row)

    # Build column names
    cols = ["id", "filename"] + [f"f{i+1}" for i in range(max_len)]

    df_final = pd.DataFrame(all_data, columns=cols)

    return df_final

if __name__ == "__main__":
     

    df = process_folder(path_to_data)

    print(df.shape)
    print(df.head())

    df.to_csv("./Raman_ML/data/raman_data_22_06_2026.csv", index=False)

[INFO] max spectrum length = 1901
(30, 1903)
     id                                        filename         f1         f2  \
0  2166  Scan 02166(1)_Sample_26_06_18_16_06_09_350.csv   8152.708   8067.539   
1  2148  Scan 02148(1)_Sample_26_06_18_14_58_54_485.csv  10992.330  11076.100   
2  2151  Scan 02151(1)_Sample_26_06_18_14_58_05_784.csv  11002.970  10966.990   
3  2163  Scan 02163(1)_Sample_26_06_18_16_05_19_018.csv   5129.645   5092.882   
4  2150  Scan 02150(1)_Sample_26_06_18_14_58_23_212.csv   5525.646   5409.627   

          f3         f4         f5         f6         f7         f8  ...  \
0   7981.521   7890.976   7792.248   7682.265   7565.390   7451.246  ...   
1  11150.890  11210.220  11247.620  11256.380  11225.710  11141.950  ...   
2  10927.980  10891.790  10864.200  10850.060  10842.070  10824.310  ...   
3   5050.128   5007.791   4972.233   4949.437   4940.929   4945.073  ...   
4   5312.078   5245.381   5221.814   5252.308   5330.977   5440.034  ...   

      f1892

#### Explore

In [ ]:
df = process_folder(path_to_data)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_spectrum(df, target_id):
    """
    Plots spectrum for a given ID from wide dataframe
    """

    # get row
    row = df[df["id"] == str(target_id)]

    if row.empty:
        print(f"[ERROR] ID {target_id} not found")
        return

    # extract intensity values (skip id + filename)
    y = row.iloc[0, 2:].values.astype(float)

    # remove NaNs (from padding)
    mask = ~np.isnan(y)
    y = y[mask]

    # x = index (since you don't care about wavelength)
    x = np.arange(len(y))

    # plot
    plt.figure(figsize=(8, 4))
    plt.plot(x, y)
    plt.title(f"ID: {target_id}")
    plt.xlabel("Index")
    plt.ylabel("Intensity")
    plt.tight_layout()
    plt.show()
 

In [ ]:
plot_spectrum(df, '2137') # pure paracetamol

In [ ]:
plot_spectrum(df, '2140') # pure glucose

In [ ]:
plot_spectrum(df, '2143')   

In [ ]:
plot_spectrum(df, '2146')

#### Add GT to data

In [ ]:
import pandas as pd


def merge_ground_truth(df_spectra, gt_path):
    """
    Merge Raman wide dataframe with ground truth CSV (from file path)

    Parameters
    ----------
    df_spectra : pandas.DataFrame
        Output from process_folder()
        columns: id, filename, f1, f2, ...

    gt_path : str
        Path to ground truth CSV file

    Returns
    -------
    pandas.DataFrame
        Fully merged dataframe
    """

    # ---------- Load ground truth ----------
    df_gt = pd.read_csv(gt_path)

    # ---------- Fix ID formatting ----------
    # spectra IDs already clean (from earlier fix), but enforce
    df_spectra = df_spectra.copy()
    df_spectra["ID"] = df_spectra["id"].astype(str).str.strip()

    # ground truth IDs → remove any leading zeros just in case
    df_gt["ID"] = df_gt["ID"].astype(str).str.strip().apply(lambda x: str(int(x)))

    # ---------- Clean text fields ----------
    for col in df_gt.columns:
        if df_gt[col].dtype == object:
            df_gt[col] = (
                df_gt[col]
                .fillna("")
                .astype(str)
                .str.strip()
            )

    # replace "0" with empty (your encoding)
    component_cols = [
        "Component1", "Component2", "Component3",
        "Component4", "Component5", "Component6"
    ]

    for col in component_cols:
        if col in df_gt.columns:
            df_gt[col] = df_gt[col].replace("0", "")

    # ---------- Merge ----------
    df_merged = pd.merge(
        df_gt,
        df_spectra,
        on="ID",
        how="inner"
    )

    # optional: drop duplicate id column
    df_merged = df_merged.drop(columns=["id"])

    return df_merged

In [ ]:
 df_ml = merge_ground_truth(df, gt_path)

In [ ]:
df_ml.to_csv('./Raman_ML/data/raman_data_gt_22_06_2026.csv')

### ML model 

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier


# -------------------------------
# 1. Prepare data
# -------------------------------

def prepare_data(df):
    df = df.copy()

    # ---- clean column names ----
    df.columns = (
        df.columns
        .str.strip()
        .str.replace("\n", " ")
        .str.replace("\r", " ")
    )

    #print("\n[DEBUG] Columns:")
    #print(df.columns.tolist())

    # ---- detect target column ----
    target_candidates = [c for c in df.columns if "primary" in c.lower()]
    
    if len(target_candidates) == 0:
        raise ValueError("No Primary component column found")

    target_col = target_candidates[0]
    print(f"[INFO] Using target column: {target_col}")

    # ---- create binary target ----
    df["target"] = (
        df[target_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("paracetamol")
    ).astype(int)

    # ---- feature columns ----
    feature_cols = [c for c in df.columns if c.startswith("f")]

    print(f"[INFO] Number of features: {len(feature_cols)}")

    # ---- convert to numeric (safe) ----
    X = df[feature_cols].apply(pd.to_numeric, errors="coerce").values

    # ✅ instead of dropping → FIX NaNs
    nan_count = np.isnan(X).sum()
    #print(f"[INFO] Total NaNs in X: {nan_count}")

    # ---- replace NaNs with 0 ----
    X = np.nan_to_num(X, nan=0.0).astype(np.float32)

    y = df["target"].values

    #print(f"[INFO] Final usable samples: {len(y)}")

    # ---- normalize spectra ----
    max_vals = np.max(X, axis=1, keepdims=True)
    max_vals[max_vals == 0] = 1   # avoid divide by zero
    X = X / max_vals

    return X, y

# -------------------------------
# 2. Train-test split
# -------------------------------

def split_data(X, y):
    return train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )


# -------------------------------
# 3. Train model
# -------------------------------

def train_model(model, X_train, y_train, scale=False):
    scaler = None

    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)

    model.fit(X_train, y_train)
    return model, scaler


# -------------------------------
# 4. Evaluate model
# -------------------------------

def evaluate_model(model, X_test, y_test, scaler=None):
    
    if scaler is not None:
        X_test = scaler.transform(X_test)

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return cm


# -------------------------------
# 5. Pipeline runner
# -------------------------------

def run_pipeline(df, model, scale=False, model_name="model"):

    print(f"\n===== {model_name.upper()} =====")

    X, y = prepare_data(df)

    X_train, X_test, y_train, y_test = split_data(X, y)

    trained_model, scaler = train_model(model, X_train, y_train, scale)

    cm = evaluate_model(trained_model, X_test, y_test, scaler)

    # Save model and scaler
    model_bundle = {
        "model": trained_model,
        "scaler": scaler
    }

    with open(f"{model_name}.pkl", "wb") as f:
        pickle.dump(model_bundle, f)

    print(f"[INFO] Saved model to {model_name}.pkl")

    return trained_model, cm


# -------------------------------
# 6. RUN
# -------------------------------

if __name__ == "__main__":

    # IMPORTANT: use MERGED dataset
    df = pd.read_csv("./Raman_ML/data/raman_data_gt_22_06_2026.csv")

    print(f"[INFO] Dataset shape: {df.shape}")

    # ---- SVM ----
    svm_model = SVC(kernel="rbf", probability=True)
    run_pipeline(df, svm_model, scale=True, model_name="SVM")

    # ---- XGBOOST ----
    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss"
    )

    run_pipeline(df, xgb_model, scale=False, model_name="XGBoost")
